In [29]:
using LowLevelFEM
using LinearAlgebra
using Tensors
using Plots

In [30]:
function cube_mesh(; lx=1.0, ly=1.0, lz=1.0, n=10, dx=lx / n, dy=ly / n, dz=lz / n, order=1)

    gmsh.option.setNumber("General.Verbosity", 0)

    # --------------------------------------------------
    # Geometry
    # --------------------------------------------------
    # Box: origin (0,0,0), size lx x ly x lz
    box = gmsh.model.occ.addBox(0.0, 0.0, 0.0, lx, ly, lz)

    gmsh.model.occ.synchronize()

    # --------------------------------------------------
    # Physical groups
    # --------------------------------------------------
    gmsh.model.addPhysicalGroup(2, [1], -1, "left")
    gmsh.model.addPhysicalGroup(2, [2], -1, "right")
    gmsh.model.addPhysicalGroup(2, [6], -1, "front")
    gmsh.model.addPhysicalGroup(2, [5], -1, "rear")
    gmsh.model.addPhysicalGroup(2, [3], -1, "bottom")
    gmsh.model.addPhysicalGroup(2, [4], -1, "top")

    gmsh.model.addPhysicalGroup(3, [1], -1, "volume")

    # --------------------------------------------------
    # Mesh settings (structured hex mesh)
    # --------------------------------------------------
    gmsh.option.setNumber("Mesh.RecombineAll", 1)

    d = min(dx, dy)

    gmsh.model.mesh.setTransfiniteCurve(9, ceil(lx / d) + 1)
    gmsh.model.mesh.setTransfiniteCurve(10, ceil(lx / d) + 1)
    gmsh.model.mesh.setTransfiniteCurve(11, ceil(lx / d) + 1)
    gmsh.model.mesh.setTransfiniteCurve(12, ceil(lx / d) + 1)

    gmsh.model.mesh.setTransfiniteCurve(2, ceil(ly / d) + 1)
    gmsh.model.mesh.setTransfiniteCurve(4, ceil(ly / d) + 1)
    gmsh.model.mesh.setTransfiniteCurve(6, ceil(ly / d) + 1)
    gmsh.model.mesh.setTransfiniteCurve(8, ceil(ly / d) + 1)

    gmsh.model.mesh.setTransfiniteCurve(1, ceil(lz / d) + 1)
    gmsh.model.mesh.setTransfiniteCurve(3, ceil(lz / d) + 1)
    gmsh.model.mesh.setTransfiniteCurve(5, ceil(lz / d) + 1)
    gmsh.model.mesh.setTransfiniteCurve(7, ceil(lz / d) + 1)

    gmsh.model.mesh.setTransfiniteSurface(1)
    gmsh.model.mesh.setTransfiniteSurface(2)
    gmsh.model.mesh.setTransfiniteSurface(3)
    gmsh.model.mesh.setTransfiniteSurface(4)
    gmsh.model.mesh.setTransfiniteSurface(5)
    gmsh.model.mesh.setTransfiniteSurface(6)

    gmsh.model.mesh.setTransfiniteVolume(1)

    gmsh.option.setNumber("Mesh.ElementOrder", order)

    # --------------------------------------------------
    # Generate mesh
    # --------------------------------------------------
    gmsh.model.mesh.generate(3)

    # --------------------------------------------------
    # Save
    # --------------------------------------------------
    #gmsh.write("cube.msh")

    return nothing
end


cube_mesh (generic function with 1 method)

In [31]:
gmsh.initialize()

In [32]:
cube_mesh(lx=10, n=4, order=1)

In [33]:
mat = Material("volume", E=2e2, ν=0.49)
prob = Problem([mat], type=:VectorField, fieldName=:u, rhsName=:f);

In [34]:
suppX = BoundaryCondition("left", ux=0, uy=0, uz=0)

load = BoundaryCondition("right", fy=(x, y, z) -> -(z - 0.5) * 1, fz=(x, y, z) -> (y - 0.5) * 1)

BoundaryCondition("right", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:fy => var"#25#26"(), :fz => var"#27#28"()))

In [35]:
f_ext = loadVector(prob, [load])

showDoFResults(f_ext, name="load");

In [36]:
μ = mat.μ
λ = mat.λ
κ = mat.κ

p = (μ=μ, λ=λ, κ=κ);

In [37]:
u = vectorField(prob, "volume", [0, 0, 0]);

In [38]:
F = tensorField(prob, "volume", [1 0 0; 0 1 0; 0 0 1])

I = unitTensor(F)

C = F' * F;

In [39]:
function ψ_StVenantKirchhoff(C, p)
    λ = p.λ
    μ = p.μ

    I = [1.0 0 0; 0 1 0; 0 0 1]

    E = (C - I) / 2

    return (λ * tr(E)^2 + 2μ * dot(E, E)) / 2
end


ψ_StVenantKirchhoff (generic function with 1 method)

In [40]:
function ψ_NeoHooke(C, p)
    μ = p.μ
    κ = p.κ

    I = [1.0 0 0; 0 1 0; 0 0 1]
    C_I = tr(C)
    C_III = det(C)
    J = √C_III

    return μ / 2 * (C_I / (C_III^(1 / 3)) - 3) + κ / 2 * log(J)^2
    #return μ / 2 * (C_I / (C_III^(1 / 3)) - 3) + κ / 2 * (J - 1)^2
end


ψ_NeoHooke (generic function with 1 method)

In [43]:
#ψ = ψ_StVenantKirchhoff
ψ = ψ_NeoHooke

ψ_NeoHooke (generic function with 1 method)

In [45]:
S = IIPiolaKirchhoffStress(ψ, C, p)

P = F * S;

In [ ]:
loadSteps = 100
iterationSteps = 20

λ = 0.0
Δλ = 1.0 / loadSteps
Δλ_min = 1e-4
Δλ_max = 2.0 / loadSteps
err_limit = 1e-3
Δu = nothing
nstep = nothing
ε = Float64[]
uu = VectorField[]

push!(uu, u)

i = 1
while λ < 1.0
    u_old = copy(u)
    F_old = copy(F)
    C_old = copy(C)

    λ_trial = min(λ + Δλ, 1.0)

    converged = false
    err = 1e10
    err_prev = Inf

    print(i, ": ", round(λ_trial * 100, digits=2), "%-(")
    j = 0
    while err < 10 * err_prev && j < iterationSteps
        j += 1
        print(j)
        sc = 30
        load = LoadCondition("right", fy=(x, y, z) -> -(z - 0.5) * sc * λ_trial, fz=(x, y, z) -> (y - 0.5) * sc * λ_trial)

        Kmat = materialTangentMatrix(prob, F=F, energy=ψ, params=p)
        Kgeo = initialStressMatrix(prob, energy=ψ, params=p, C=C)
        Kint = Kmat + Kgeo

        Fright = nodesToElements(elementsToNodes(F), onPhysicalGroup="right")
        Kext = externalTangentFollower(prob, [load], F=Fright)
        K = Kint - Kext

        f_int = internalForceVector(prob, energy=ψ, params=p, F=F)
        f_ext = loadVector(prob, [load], F=Fright)
        f = f_ext - f_int

        Δu = solveField(K, f, support=[suppX])

        u += Δu
        F = I + u ∘ ∇
        C = F' * F
        S = IIPiolaKirchhoffStress(ψ, C, p)
        P = F * S

        err = norm(DoFs(Δu)[:, 1]) / max(norm(DoFs(u)[:, 1]), 1e-12)

        push!(ε, err)

        if err < err_limit
            converged = true
            println(")✓")
            i += 1
            nstep = i
            break
        end

        if err > err_prev
            break
        end

        print(",")

        err_prev = err
    end
    if converged
        λ = λ_trial
        push!(uu, u)
        Δλ = min(1.1 * Δλ, Δλ_max)
    else
        u = copy(u_old)
        F = copy(F_old)
        C = copy(C_old)
        Δλ /= 1.3
        Δλ = clamp(Δλ, Δλ_min, Δλ_max)
        println(")↯")
    end
end
println()

1: 

1.0%-(1,2,3)✓
2: 2.1%-(1,2,3)↯
2: 1.85%-(1,2,3)↯
2: 1.65%-(1,2,3)↯
2: 1.5%-(1,2,3)↯
2: 1.39%-(1,2,3)↯
2: 1.3%-(1,2,3)↯
2: 1.23%-(1,2,3)↯
2: 1.18%-(1,2,3)↯
2: 1.13%-(1,2,3)✓
3: 1.28%-(1,2,3,4)✓
4: 1.45%-(1,2,3,4)✓
5: 1.63%-(1,2,3)↯
5: 1.58%-(1,2,3,4)✓
6: 1.74%-(1,2,3,4)✓
7: 1.9%-(1,2,3,4)✓
8: 2.09%-(1,2,3)↯
8: 2.04%-(1,2,3,4)✓
9: 2.2%-(1,2,3,4)✓
10: 2.37%-(1,2,3,4)✓
11: 2.56%-(1,2,3)↯
11: 2.52%-(1,2,3,4)✓
12: 2.68%-(1,2,3,4)✓
13: 2.85%-(1,2,3,4)✓
14: 3.04%-(1,2,3)↯
14: 3.0%-(1,2,3,4)✓
15: 3.16%-(1,2,3,4)✓
16: 3.34%-(1,2,3,4)✓
17: 3.54%-(1,2,3,4)✓
18: 3.75%-(1,2,3)↯
18: 3.7%-(1,2,3,4)✓
19: 3.89%-(1,2,3,4)✓
20: 4.09%-(1,2,3,4,5)✓
21: 4.31%-(1,2,3)↯
21: 4.26%-(1,2,3,4,5)✓
22: 4.45%-(1,2,3,4,5)✓
23: 4.66%-(1,2,3,4,5)✓
24: 4.88%-(1,2,3,4,5)✓
25: 5.13%-(1,2,3)↯
25: 5.08%-(1,2,3,4,5)✓
26: 5.29%-(1,2,3,4,5)✓
27: 5.52%-(1,2,3,4,5)✓
28: 5.78%-(1,2,3,4,5,6)✓
29: 6.06%-(1,2,3)↯
29: 5.99%-(1,2,3,4,5)✓
30: 6.23%-(1,2,3,4,5,6)✓
31: 6.49%-(1,2,3,4,5,6)✓
32: 6.78%-(1,2,3,4,5,6)✓
33: 7.1%-(1,2,3,4,5,6,7)

In [ ]:
plot(ε, yscale=:log10, label="ε")

In [ ]:
u = mergeFields(uu);

In [ ]:
showDoFResults(u, name="u", visible=true);

In [ ]:
I = ScalarField(prob, "volume", 1.0, steps=nstep)
O = ScalarField(prob, "volume", 0.0, steps=nstep)
I = TensorField([I O O; O I O; O O I])
F = I + u ∘ ∇
C = F' * F
S = IIPiolaKirchhoff(ψ, C, p);

In [ ]:
showDoFResults(S, name="II P-K");

In [ ]:
openPostProcessor()

In [ ]:
saveField("u", u)

In [ ]:
gmsh.finalize()